# 01 — Exploratory Data Analysis
**Projet :** Daily Electricity Load Forecasting — Québec Grid  
**Objectif :** Comprendre la structure des données, identifier les patterns saisonniers et hebdomadaires, détecter les valeurs aberrantes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("Librairies chargées ✓")

## 1. Chargement des données

In [ ]:
# Après avoir lancé data/download_data.py, le fichier est dans data/raw/load_daily.csv
# Pour l'instant on charge le CSV de développement
df = pd.read_csv("quebec_load_daily.csv", parse_dates=["date"])

print(f"Shape         : {df.shape}")
print(f"Période       : {df.date.min().date()} → {df.date.max().date()}")
print(f"Valeurs nulles: {df.isnull().sum().sum()}")
df.head(10)

## 2. Statistiques descriptives

In [ ]:
desc = df["load_GWh"].describe().rename("load_GWh")
print(desc.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution
axes[0].hist(df["load_GWh"], bins=50, color="#4C72B0", edgecolor="white")
axes[0].axvline(df["load_GWh"].mean(), color="crimson", linestyle="--", label=f"Moyenne : {df['load_GWh'].mean():.0f} GWh")
axes[0].set_title("Distribution de la consommation journalière")
axes[0].set_xlabel("GWh/jour")
axes[0].set_ylabel("Nombre de jours")
axes[0].legend()

# Boxplot par mois
df["month"] = df["date"].dt.month
monthly = [df[df["month"] == m]["load_GWh"].values for m in range(1, 13)]
axes[1].boxplot(monthly, labels=["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"])
axes[1].set_title("Distribution mensuelle")
axes[1].set_xlabel("Mois")
axes[1].set_ylabel("GWh/jour")

plt.tight_layout()
plt.savefig("fig_distributions.png", bbox_inches="tight")
plt.show()

## 3. Série temporelle complète

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df["date"], df["load_GWh"], linewidth=0.7, color="#4C72B0", alpha=0.8)

# Moyenne mobile 30 jours
rolling = df["load_GWh"].rolling(30, center=True).mean()
ax.plot(df["date"], rolling, color="crimson", linewidth=2, label="Moyenne mobile 30j")

ax.set_title("Consommation électrique journalière — Réseau Québec (2018–2023)")
ax.set_xlabel("Date")
ax.set_ylabel("GWh/jour")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()

plt.tight_layout()
plt.savefig("fig_timeseries.png", bbox_inches="tight")
plt.show()

## 4. Patterns saisonniers et hebdomadaires

In [ ]:
df["dayofweek"] = df["date"].dt.dayofweek  # 0=Lundi, 6=Dimanche
df["year"] = df["date"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Pattern hebdomadaire
jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
weekly_avg = df.groupby("dayofweek")["load_GWh"].mean()
axes[0].bar(jours, weekly_avg.values, color=sns.color_palette("muted", 7))
axes[0].set_title("Consommation moyenne par jour de la semaine")
axes[0].set_ylabel("GWh/jour")

# Pattern saisonnier (courbe par année)
df["dayofyear"] = df["date"].dt.dayofyear
for year in df["year"].unique():
    subset = df[df["year"] == year].sort_values("dayofyear")
    axes[1].plot(subset["dayofyear"], subset["load_GWh"], linewidth=0.8, alpha=0.6, label=str(year))
axes[1].set_title("Saisonnalité par année (jour julien)")
axes[1].set_xlabel("Jour de l'année")
axes[1].set_ylabel("GWh/jour")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("fig_patterns.png", bbox_inches="tight")
plt.show()

# Résumé numérique
print("\nConsommation moyenne par jour de semaine:")
print(df.groupby("dayofweek")["load_GWh"].mean().rename(index=dict(enumerate(jours))).to_string())

## 5. Autocorrélation — pertinence des lag features

In [ ]:
from pandas.plotting import autocorrelation_plot

lags = [1, 2, 7, 14, 21, 28]
corrs = [df["load_GWh"].autocorr(lag=l) for l in lags]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart des corrélations aux lags clés
axes[0].bar([str(l) for l in lags], corrs, color="#4C72B0")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("Autocorrélation aux lags clés")
axes[0].set_xlabel("Lag (jours)")
axes[0].set_ylabel("Coefficient de corrélation")
for i, (lag, c) in enumerate(zip(lags, corrs)):
    axes[0].text(i, c + 0.01, f"{c:.2f}", ha="center", fontsize=9)

# Autocorrelation plot complet (60 jours)
pd.plotting.autocorrelation_plot(df["load_GWh"], ax=axes[1])
axes[1].set_xlim(0, 60)
axes[1].set_title("Autocorrélation complète (0–60 jours)")

plt.tight_layout()
plt.savefig("fig_autocorr.png", bbox_inches="tight")
plt.show()

print("\nConclusion : les lags J-1, J-7 et J-14 sont fortement corrélés → bons candidats pour le feature engineering.")

## 6. Détection des valeurs aberrantes

In [ ]:
Q1 = df["load_GWh"].quantile(0.25)
Q3 = df["load_GWh"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["load_GWh"] < lower) | (df["load_GWh"] > upper)]
print(f"Seuil bas  : {lower:.0f} GWh/jour")
print(f"Seuil haut : {upper:.0f} GWh/jour")
print(f"Outliers détectés : {len(outliers)} ({len(outliers)/len(df)*100:.1f}% des jours)")
if len(outliers) > 0:
    print("\nDétail:")
    print(outliers[["date", "load_GWh"]].to_string(index=False))
else:
    print("\nAucune valeur aberrante détectée — données propres ✓")

## 7. Synthèse — Observations clés

| Observation | Implication pour le modèle |
|---|---|
| **Forte saisonnalité hivernale** | Inclure le mois et le jour de l'année comme features |
| **Pattern hebdomadaire clair** (weekends -4%) | Inclure le jour de la semaine |
| **Autocorrélation forte à J-1, J-7, J-14** | Lag features prioritaires |
| **Données propres, peu d'outliers** | Pas de traitement particulier nécessaire |

**Prochaine étape →** `02_features.ipynb` : construire le pipeline de feature engineering.